# Session1_Task5 — Customer Analysis

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# โหลด customers_cleaned.csv (ผ่าน Task 2 แล้ว)
c = pd.read_csv('customers_cleaned.csv')

In [2]:
# --- 1. Age Groups ---

# pd.cut() → แบ่งข้อมูลตัวเลขออกเป็นกลุ่มตามช่วงที่กำหนด
#   bins   → ขอบเขตของแต่ละกลุ่ม
#   labels → ชื่อของแต่ละกลุ่ม
#   right=True → ขอบขวา inclusive เช่น (18,24] = 18 < x <= 24
c['age_group'] = pd.cut(
    c['age'],
    bins=[17, 24, 34, 44, 200],
    labels=['18-24', '25-34', '35-44', '45+']
)

# .value_counts() → นับจำนวนในแต่ละกลุ่ม
# .sort_index()   → เรียงตามชื่อ label
age_counts = c['age_group'].value_counts().sort_index()
print('Age groups:\n', age_counts)

Age groups:
 age_group
18-24     78
25-34    118
35-44    123
45+      401
Name: count, dtype: int64


In [3]:
# --- 2. Gender Distribution ---

# กรองเฉพาะ M และ F (ตาม data dictionary)
# .isin() → เช็คว่าค่าอยู่ใน list หรือไม่
gender_counts = c[c['gender'].isin(['M','F'])]['gender'].value_counts()

# คำนวณ % โดย หาร sum แล้วคูณ 100
gender_pct = (gender_counts / gender_counts.sum() * 100).round(2)
gender_df = pd.DataFrame({'Gender': gender_pct.index, 'Percentage (%)': gender_pct.values})
print('\nGender distribution:')
display(gender_df)


Gender distribution:


,Gender,Percentage (%)
0,M,50.77
1,F,49.23


In [4]:
# --- 3. Average Spending per Loyalty Tier ---

# .str.title() → แปลงเป็น Title Case (Basic, Silver, Gold) แก้ความไม่สม่ำเสมอ
c['membership_status'] = c['membership_status'].str.title()

# กรองเฉพาะ 3 tier หลัก
tier = c[c['membership_status'].isin(['Basic','Silver','Gold'])]

# .groupby().agg() → จัดกลุ่มและคำนวณค่าเฉลี่ย
avg_spend = (tier.groupby('membership_status')['total_spending']
                 .mean().round(2)
                 .reindex(['Basic','Silver','Gold']))  # .reindex() → จัดลำดับ tier
tier_df = pd.DataFrame({'Tier': avg_spend.index, 'Avg Spending ($)': avg_spend.values})
print('\nAverage spending per tier:')
display(tier_df)


Average spending per tier:


,Tier,Avg Spending ($)
0,Basic,44130.41
1,Silver,83806.55
2,Gold,72854.51


In [5]:
# --- สร้าง PDF ---
with PdfPages('Session1_CustomerAnalysis.pdf') as pdf:

    # หน้า 1: Bar Chart - Age Groups
    fig, ax = plt.subplots(figsize=(8, 5))
    age_counts.plot(kind='bar', color='steelblue', ax=ax, rot=0)
    ax.set_title('Distribution of Customer Age Groups', fontsize=14, fontweight='bold', pad=12)
    ax.set_ylabel('Number of Customers')
    ax.set_xlabel('Age Group')
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    fig.tight_layout()
    pdf.savefig(fig, bbox_inches='tight'); plt.close()

    # หน้า 2: Gender Table
    fig, ax = plt.subplots(figsize=(6, 2))
    ax.axis('off')
    ax.set_title('Gender Distribution (%)', fontsize=13, fontweight='bold', pad=16)
    tbl = ax.table(cellText=gender_df.values, colLabels=gender_df.columns, loc='center', cellLoc='center')
    tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1.5, 2.2)
    for (r, _), cell in tbl.get_celld().items():
        if r == 0: cell.set_facecolor('steelblue'); cell.set_text_props(color='white', fontweight='bold')
        cell.set_edgecolor('lightgray')
    fig.tight_layout()
    pdf.savefig(fig, bbox_inches='tight'); plt.close()

    # หน้า 3: Loyalty Tier Table
    fig, ax = plt.subplots(figsize=(6, 2))
    ax.axis('off')
    ax.set_title('Average Spending per Loyalty Tier', fontsize=13, fontweight='bold', pad=16)
    tbl = ax.table(cellText=tier_df.values, colLabels=tier_df.columns, loc='center', cellLoc='center')
    tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1.5, 2.2)
    for (r, _), cell in tbl.get_celld().items():
        if r == 0: cell.set_facecolor('seagreen'); cell.set_text_props(color='white', fontweight='bold')
        cell.set_edgecolor('lightgray')
    fig.tight_layout()
    pdf.savefig(fig, bbox_inches='tight'); plt.close()

print('✅ Saved Session1_CustomerAnalysis.pdf')

# === จุดสังเกต ===
# ✔ Age group ครบ 4 กลุ่ม: 18-24, 25-34, 35-44, 45+
# ✔ Gender % รวมกันได้ 100%
# ✔ Loyalty tier มีครบ Basic, Silver, Gold

✅ Saved Session1_CustomerAnalysis.pdf
